# 🐜 Exercícios — Otimização por Colônia de Formigas (ACO)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Simule formigas construindo soluções para o TSP usando feromônios e heurísticas.


## 1. ACO para o Problema do Caixeiro Viajante

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Cidades
N_CIDADES = 12
cidades = np.random.rand(N_CIDADES, 2) * 100

# Matriz de distâncias
dist = np.zeros((N_CIDADES, N_CIDADES))
for i in range(N_CIDADES):
    for j in range(N_CIDADES):
        if i != j:
            dist[i,j] = np.linalg.norm(cidades[i]-cidades[j])

def calcular_custo(rota):
    return sum(dist[rota[i], rota[(i+1)%N_CIDADES]] for i in range(N_CIDADES))

class ACO_TSP:
    def __init__(self, n_formigas=20, alfa=1.0, beta=2.0, rho=0.5, Q=100):
        self.n_formigas = n_formigas
        self.alfa = alfa   # peso do feromônio
        self.beta = beta   # peso da heurística (1/distância)
        self.rho  = rho    # taxa de evaporação
        self.Q    = Q      # quantidade de feromônio depositado
        self.tau  = np.ones((N_CIDADES, N_CIDADES))  # feromônios iniciais
        self.eta  = np.where(dist > 0, 1.0/dist, 0)  # heurística
    
    def construir_solucao(self):
        inicio = np.random.randint(N_CIDADES)
        rota = [inicio]
        nao_visitados = set(range(N_CIDADES)) - {inicio}
        while nao_visitados:
            atual = rota[-1]
            probs = np.array([
                (self.tau[atual,j]**self.alfa) * (self.eta[atual,j]**self.beta)
                if j in nao_visitados else 0
                for j in range(N_CIDADES)
            ])
            if probs.sum() == 0:
                probs = np.array([1.0 if j in nao_visitados else 0 for j in range(N_CIDADES)])
            probs /= probs.sum()
            prox = np.random.choice(N_CIDADES, p=probs)
            rota.append(prox)
            nao_visitados.remove(prox)
        return rota
    
    def atualizar_feromonios(self, solucoes):
        self.tau *= (1 - self.rho)  # evaporação
        for rota, custo in solucoes:
            delta = self.Q / custo
            for i in range(N_CIDADES):
                a, b = rota[i], rota[(i+1)%N_CIDADES]
                self.tau[a,b] += delta
                self.tau[b,a] += delta
    
    def executar(self, iteracoes=100):
        melhor_rota = None; melhor_custo = float('inf')
        historico = []
        for it in range(iteracoes):
            solucoes = []
            for _ in range(self.n_formigas):
                rota = self.construir_solucao()
                custo = calcular_custo(rota)
                solucoes.append((rota, custo))
                if custo < melhor_custo:
                    melhor_custo = custo; melhor_rota = rota[:]
            self.atualizar_feromonios(solucoes)
            historico.append(melhor_custo)
        return melhor_rota, melhor_custo, historico

aco = ACO_TSP(n_formigas=20, alfa=1.0, beta=3.0, rho=0.4)
melhor_rota, melhor_custo, hist_aco = aco.executar(iteracoes=150)

print(f"Melhor rota: {melhor_rota}")
print(f"Custo total: {melhor_custo:.2f}")

fig, axes = plt.subplots(1,2,figsize=(12,5))
# Rota
ax=axes[0]
rota_f = melhor_rota + [melhor_rota[0]]
ax.plot(cidades[rota_f,0],cidades[rota_f,1],'b-o',markersize=8)
for i,c in enumerate(cidades): ax.annotate(str(i),(c[0]+0.5,c[1]+0.5),fontsize=9)
ax.set_title(f'Melhor Rota ACO (custo={melhor_custo:.1f})')
# Convergência
axes[1].plot(hist_aco,'g-'); axes[1].set_title('Convergência ACO'); axes[1].set_xlabel('Iteração'); axes[1].grid(True)
plt.tight_layout(); plt.show()


### 📝 Exercício 1

Experimente com diferentes valores de **α (alfa)** e **β (beta)**:
- α alto = mais importância ao feromônio
- β alto = mais importância à distância

Qual combinação converge mais rápido para boas soluções?

In [ ]:
configs_aco = [
    (1.0, 1.0, "α=1, β=1 (balanceado)"),
    (1.0, 5.0, "α=1, β=5 (heurística domina)"),
    (3.0, 1.0, "α=3, β=1 (feromônio domina)"),
    (2.0, 3.0, "α=2, β=3 (heurística ligeiramente)"),
]
plt.figure(figsize=(10,4))
for alfa, beta, label in configs_aco:
    np.random.seed(42)
    aco_t = ACO_TSP(n_formigas=20, alfa=alfa, beta=beta, rho=0.4)
    _, _, hist = aco_t.executar(iteracoes=100)
    plt.plot(hist, label=label)
plt.xlabel('Iteração'); plt.ylabel('Melhor Custo'); plt.title('Impacto de α e β no ACO')
plt.legend(); plt.grid(True); plt.show()


## 2. Visualizando os Feromônios

Veja como os feromônios se concentram nas melhores arestas ao longo do tempo.

In [ ]:
np.random.seed(42)
aco_vis = ACO_TSP(n_formigas=20, alfa=1.0, beta=3.0, rho=0.4)

# Capturar feromônios em diferentes momentos
momentos = {}
for it in range(100):
    solucoes = []
    for _ in range(aco_vis.n_formigas):
        rota = aco_vis.construir_solucao()
        custo = calcular_custo(rota)
        solucoes.append((rota, custo))
    aco_vis.atualizar_feromonios(solucoes)
    if it in [0, 9, 29, 99]:
        momentos[it+1] = aco_vis.tau.copy()

fig, axes = plt.subplots(2,2,figsize=(12,10))
for ax, (it, tau) in zip(axes.flat, momentos.items()):
    tau_norm = (tau - tau.min())/(tau.max()-tau.min()+1e-10)
    for i in range(N_CIDADES):
        for j in range(i+1, N_CIDADES):
            lw = tau_norm[i,j]*5 + 0.1
            alpha = float(tau_norm[i,j])*0.8+0.1
            ax.plot([cidades[i,0],cidades[j,0]],[cidades[i,1],cidades[j,1]],
                    'b-', linewidth=lw, alpha=alpha)
    ax.scatter(cidades[:,0],cidades[:,1],c='red',s=60,zorder=5)
    for k,c in enumerate(cidades): ax.annotate(str(k),(c[0]+0.5,c[1]+0.5),fontsize=8)
    ax.set_title(f'Feromônios — Iteração {it}'); ax.axis('equal')
plt.suptitle('Evolução dos Feromônios no ACO', fontsize=13)
plt.tight_layout(); plt.show()


### 📝 Exercício Final

Modifique o ACO para usar a estratégia **Ant System Elitista (EAS)**: além do depósito normal, deposite feromônio extra na **melhor rota global** a cada iteração. Essa modificação melhora a convergência?

In [ ]:
class ACO_EAS(ACO_TSP):
    """Ant System Elitista."""
    def __init__(self, *args, e=3, **kwargs):
        super().__init__(*args, **kwargs)
        self.e = e  # fator elitista
        self.melhor_global = None
        self.custo_global = float('inf')
    
    def atualizar_feromonios(self, solucoes):
        self.tau *= (1 - self.rho)
        for rota, custo in solucoes:
            delta = self.Q / custo
            for i in range(N_CIDADES):
                a,b = rota[i], rota[(i+1)%N_CIDADES]
                self.tau[a,b]+=delta; self.tau[b,a]+=delta
        # Bônus elitista para a melhor rota global
        if self.melhor_global:
            delta_e = self.e * self.Q / self.custo_global
            for i in range(N_CIDADES):
                a,b = self.melhor_global[i], self.melhor_global[(i+1)%N_CIDADES]
                self.tau[a,b]+=delta_e; self.tau[b,a]+=delta_e
    
    def executar(self, iteracoes=100):
        melhor_rota=None; melhor_custo=float('inf'); hist=[]
        for it in range(iteracoes):
            sols=[]
            for _ in range(self.n_formigas):
                rota=self.construir_solucao(); custo=calcular_custo(rota)
                sols.append((rota,custo))
                if custo<melhor_custo: melhor_custo=custo; melhor_rota=rota[:]
            self.melhor_global=melhor_rota; self.custo_global=melhor_custo
            self.atualizar_feromonios(sols); hist.append(melhor_custo)
        return melhor_rota, melhor_custo, hist

np.random.seed(42)
aco_eas = ACO_EAS(n_formigas=20, alfa=1.0, beta=3.0, rho=0.4, e=5)
_, _, hist_eas = aco_eas.executar(150)

np.random.seed(42)
aco_std = ACO_TSP(n_formigas=20, alfa=1.0, beta=3.0, rho=0.4)
_, _, hist_std = aco_std.executar(150)

plt.figure(figsize=(9,4))
plt.plot(hist_std,'b-',label='ACO padrão')
plt.plot(hist_eas,'r-',label='ACO Elitista (EAS)')
plt.xlabel('Iteração'); plt.ylabel('Melhor Custo'); plt.title('ACO vs ACO-EAS')
plt.legend(); plt.grid(True); plt.show()
